In [ ]:
import os
import cv2
import numpy as np
import shutil
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
from tensorflow.keras.optimizers import Adam

# -----------------------------
# CONFIGURATION
# -----------------------------
IMG_SIZE = 200

# Folders where UTKFace data is stored
SOURCE_FOLDERS = [
    r"C:\Users\ashid\Downloads\archive (1)\UTKFace",
    r"C:\Users\ashid\Downloads\archive (1)\crop_part1",
    r"C:\Users\ashid\Downloads\archive (1)\utkface_aligned_cropped"
]

# Unified destination folder
DEST_FOLDER = r"C:\Users\ashid\Documents\all_utkface"

# -----------------------------
# FUNCTION: Combine all images
# -----------------------------
def combine_folders(source_folders, dest_folder):
    os.makedirs(dest_folder, exist_ok=True)
    for folder in source_folders:
        if os.path.exists(folder):
            for file in os.listdir(folder):
                src_file = os.path.join(folder, file)
                dst_file = os.path.join(dest_folder, file)
                if os.path.isfile(src_file):
                    shutil.copy(src_file, dst_file)
    print(f"✅ All images copied to: {dest_folder}")

# -----------------------------
# FUNCTION: Load and preprocess images
# -----------------------------
def load_data(data_dir):
    images, ages, genders = [], [], []
    
    for img_name in os.listdir(data_dir):
        try:
            parts = img_name.split('_')
            age = int(parts[0])
            gender = int(parts[1])
            img_path = os.path.join(data_dir, img_name)
            img = cv2.imread(img_path)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)
            ages.append(age)
            genders.append(gender)
        except:
            continue

    images = np.array(images) / 255.0
    ages = np.array(ages)
    genders = np.array(genders)
    return train_test_split(images, ages, genders, test_size=0.2)

# -----------------------------
# FUNCTION: Build CNN Model
# -----------------------------
def build_model():
    input_layer = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    
    x = Conv2D(32, (3,3), activation='relu')(input_layer)
    x = MaxPooling2D((2,2))(x)
    x = Conv2D(64, (3,3), activation='relu')(x)
    x = MaxPooling2D((2,2))(x)
    x = Flatten()(x)

    gender_output = Dense(1, activation='sigmoid', name='gender')(x)
    age_output = Dense(1, activation='linear', name='age')(x)

    model = Model(inputs=input_layer, outputs=[gender_output, age_output])
    model.compile(optimizer=Adam(),
                  loss={'gender': 'binary_crossentropy', 'age': 'mse'},
                  metrics={'gender': 'accuracy', 'age': 'mae'})
    return model

# -----------------------------
# MAIN FUNCTION
# -----------------------------
def main():
    print("[INFO] Combining folders...")
    combine_folders(SOURCE_FOLDERS, DEST_FOLDER)

    print("[INFO] Loading and preprocessing data...")
    X_train, X_test, y_age_train, y_age_test, y_gender_train, y_gender_test = load_data(DEST_FOLDER)

    print("[INFO] Building model...")
    model = build_model()

    print("[INFO] Training model...")
    model.fit(X_train,
              {'gender': y_gender_train, 'age': y_age_train},
              validation_data=(X_test, {'gender': y_gender_test, 'age': y_age_test}),
              epochs=10,
              batch_size=32)

    print("[INFO] Saving model...")
    model.save("age_gender_model.h5")

    # Prediction example
    print("[INFO] Predicting on a test image...")
    idx = np.random.randint(0, len(X_test))
    sample = np.expand_dims(X_test[idx], axis=0)
    gender_pred, age_pred = model.predict(sample)

    predicted_gender = "Female" if gender_pred[0][0] > 0.5 else "Male"
    predicted_age = int(age_pred[0][0])

    plt.imshow(X_test[idx])
    plt.title(f"Predicted: {predicted_gender}, Age: {predicted_age}")
    plt.axis('off')
    plt.show()

# -----------------------------
# ENTRY POINT
# -----------------------------
if __name__ == "__main__":
    main()
